In [25]:
df = spark.read.parquet(
    "abfss://btc-transactions@btcetlstorage.dfs.core.windows.net/btc_*.parquet"
)
print(f"Total rows: {df.count()}")
print(f"Columns: {df.columns}")
df.show(5)

StatementMeta(etlsparkpool, 9, 2, Finished, Available, Finished, False)

Total rows: 8781456
Columns: ['txid', 'hash', 'version', 'size', 'block_hash', 'block_number', 'index', 'virtual_size', 'lock_time', 'input_count', 'output_count', 'is_coinbase', 'output_value', 'outputs', 'block_timestamp', 'date', 'last_modified', 'fee', 'input_value', 'inputs']
+--------------------+--------------------+-------+----+--------------------+------------+-----+------------+---------+-----------+------------+-----------+-------------------+--------------------+-------------------+----------+--------------------+------+-------------------+--------------------+
|                txid|                hash|version|size|          block_hash|block_number|index|virtual_size|lock_time|input_count|output_count|is_coinbase|       output_value|             outputs|    block_timestamp|      date|       last_modified|   fee|        input_value|              inputs|
+--------------------+--------------------+-------+----+--------------------+------------+-----+------------+---------+---

big picture

df (8.7M rows, 20 columns, raw)
        ↓
.select() → 9 important columns
        ↓
.dropna() → remove missing data
        ↓
.filter() → remove mining transactions
        ↓
.withColumn() × 3 → add fee%, complexity, category
        ↓
df_clean → ready for analysis! ✅

In [26]:
from pyspark.sql.functions import col, round, when

# Select important columns
df_clean = df.select(
    "txid",
    "block_timestamp",
    "block_number",
    "output_value",
    "input_value",
    "fee",
    "input_count",
    "output_count",
    "is_coinbase"
)

StatementMeta(etlsparkpool, 9, 3, Finished, Available, Finished, False)

removes mining reward transactions. we only want real person-to-person transfers for our analytics.

In [27]:

# Drop nulls
df_clean = df_clean.dropna(subset=["txid", "output_value", "block_timestamp"])

# Filter out coinbase transactions
df_clean = df_clean.filter(col("is_coinbase") == False)

StatementMeta(etlsparkpool, 9, 4, Finished, Available, Finished, False)

what is fee percentage?
every Bitcoin transaction pays a small fee to miners who process it. fee percentage tells us:

"what % of the total amount sent was paid as fee?"

example:

input_value = 1.0 BTC (you're sending this)
fee = 0.0001 BTC (you pay this to miners)
fee_percentage = 0.01%

Complexity ratio
tells us how many receivers per sender:

ratio = 1.0 → simple, 1 person sending to 1 person
ratio = 10.0 → complex, 1 person sending to 10 people (like a company paying employees)

In [28]:
# New column 1: fee percentage
df_clean = df_clean.withColumn(
    "fee_percentage",
    round((col("fee") / col("input_value")) * 100, 4)
)

# New column 2: complexity ratio
df_clean = df_clean.withColumn(
    "complexity_ratio",
    round(col("output_count") / col("input_count"), 4)
)

# New column 3: transaction size category
df_clean = df_clean.withColumn(
    "btc_value_category",
    when(col("output_value") > 50, "whale")
    .when(col("output_value") > 10, "large")
    .when(col("output_value") > 1, "medium")
    .otherwise("small")
)

StatementMeta(etlsparkpool, 9, 5, Finished, Available, Finished, False)

In [29]:
print(f"Rows after cleaning: {df_clean.count()}")
df_clean.show(5)

StatementMeta(etlsparkpool, 9, 6, Finished, Available, Finished, False)

Rows after cleaning: 8772036
+--------------------+-------------------+------------+-------------------+-------------------+------+-----------+------------+-----------+--------------+----------------+------------------+
|                txid|    block_timestamp|block_number|       output_value|        input_value|   fee|input_count|output_count|is_coinbase|fee_percentage|complexity_ratio|btc_value_category|
+--------------------+-------------------+------------+-------------------+-------------------+------+-----------+------------+-----------+--------------+----------------+------------------+
|7cdd3ad4ab269d2f4...|2014-07-01 03:24:17|      308684|         0.04980851|         0.04990851|1.0E-4|          1|           2|      false|        0.2004|             2.0|             small|
|b6b7f17318b70be8f...|2014-07-01 20:10:27|      308780|            0.11257|            0.11267|1.0E-4|          1|           2|      false|        0.0888|             2.0|             small|
|3f97ec09c4c62a9

In [30]:
df_clean.groupBy("btc_value_category").count().show()

StatementMeta(etlsparkpool, 9, 7, Finished, Available, Finished, False)

+------------------+-------+
|btc_value_category|  count|
+------------------+-------+
|             whale| 307484|
|            medium|1757882|
|             small|6100875|
|             large| 605795|
+------------------+-------+



In [31]:
df_clean.filter(col("btc_value_category")=="whale").show(10)

StatementMeta(etlsparkpool, 9, 8, Finished, Available, Finished, False)

+--------------------+-------------------+------------+------------------+------------------+------+-----------+------------+-----------+--------------+----------------+------------------+
|                txid|    block_timestamp|block_number|      output_value|       input_value|   fee|input_count|output_count|is_coinbase|fee_percentage|complexity_ratio|btc_value_category|
+--------------------+-------------------+------------+------------------+------------------+------+-----------+------------+-----------+--------------+----------------+------------------+
|92b6a780944477acd...|2014-07-01 01:43:56|      308679|      100.03114487|100.03174487000001|6.0E-4|         25|          19|      false|        6.0E-4|            0.76|             whale|
|646a5a1228a62b183...|2014-07-01 21:31:50|      308789|       56.46025242|56.460752420000006|5.0E-4|          1|           3|      false|        9.0E-4|             3.0|             whale|
|59ae4d68739bf80f4...|2014-07-01 18:14:51|      308773|

In [32]:
# Count of each category
df_clean.groupBy("btc_value_category").count().orderBy("count", ascending=False).show()

StatementMeta(etlsparkpool, 9, 9, Finished, Available, Finished, False)

+------------------+-------+
|btc_value_category|  count|
+------------------+-------+
|             small|6100875|
|            medium|1757882|
|             large| 605795|
|             whale| 307484|
+------------------+-------+



🥉 BRONZE LAYER - Raw data as is

In [33]:
# 🥉 BRONZE LAYER - Raw data as is
print("🥉 Writing Bronze Layer...")

bronze_path = "abfss://btc-transactions@btcetlstorage.dfs.core.windows.net/bronze/"

df.write.mode("overwrite").parquet(bronze_path)

print(f"✅ Bronze Layer created!")
print(f"   Rows: {df.count()}")
print(f"   Columns: {len(df.columns)}")

StatementMeta(etlsparkpool, 9, 10, Finished, Available, Finished, False)

🥉 Writing Bronze Layer...
✅ Bronze Layer created!
   Rows: 8781456
   Columns: 20


🥈 SILVER LAYER - Cleaned and transformed data

In [34]:
# 🥈 SILVER LAYER - Cleaned and transformed data
print("🥈 Writing Silver Layer...")

silver_path = "abfss://btc-transactions@btcetlstorage.dfs.core.windows.net/silver/"

df_clean.write.mode("overwrite").parquet(silver_path)

print(f"✅ Silver Layer created!")
print(f"   Rows: {df_clean.count()}")
print(f"   Columns: {len(df_clean.columns)}")

StatementMeta(etlsparkpool, 9, 11, Finished, Available, Finished, False)

🥈 Writing Silver Layer...
✅ Silver Layer created!
   Rows: 8772036
   Columns: 12


🥇 GOLD LAYER - Aggregated business ready data

In [35]:
# 🥇 GOLD LAYER - Aggregated business ready data
from pyspark.sql.functions import year, count, avg, sum, max, min

print("🥇 Writing Gold Layer...")

gold_path = "abfss://btc-transactions@btcetlstorage.dfs.core.windows.net/gold/"

# Aggregate by year
df_gold = df_clean.groupBy(year("block_timestamp").alias("year")) \
    .agg(
        count("txid").alias("total_transactions"),
        avg("output_value").alias("avg_output_value"),
        sum("output_value").alias("total_output_value"),
        avg("fee_percentage").alias("avg_fee_percentage"),
        avg("complexity_ratio").alias("avg_complexity_ratio"),
        max("output_value").alias("max_transaction_value"),
        min("output_value").alias("min_transaction_value")
    ).orderBy("year")

df_gold.show()

# Save Gold layer
df_gold.write.mode("overwrite").parquet(gold_path)

print(f"✅ Gold Layer created!")
print(f"   Rows: {df_gold.count()}")

StatementMeta(etlsparkpool, 9, 12, Finished, Available, Finished, False)

🥇 Writing Gold Layer...
+----+------------------+------------------+--------------------+-------------------+--------------------+---------------------+---------------------+
|year|total_transactions|  avg_output_value|  total_output_value| avg_fee_percentage|avg_complexity_ratio|max_transaction_value|min_transaction_value|
+----+------------------+------------------+--------------------+-------------------+--------------------+---------------------+---------------------+
|2013|            568132|21.884528969221744|1.2433301212341888E7|  1.015960537515905|  1.9809125803510614|   49042.469999999994|               1.0E-8|
|2014|            792130|10.628331760592749|   8419020.437518334|  0.989555240806405|  2.0370819530885202|             39016.97|                  0.0|
|2015|           1459810|10.706786481291436|1.5629873973254051E7|  1.260694390105739|  2.7849536983580214|       92114.13874635|                  0.0|
|2016|           2628115|12.978758310803126| 3.410966939799636E7| 1.39

In [36]:
import pandas as pd

# Convert Gold layer to pandas for better display
gold_pandas = df_gold.toPandas()

# Round all numbers to 2 decimal places
gold_pandas = gold_pandas.round(2)

# Rename columns for better readability
gold_pandas.columns = [
    'Year', 
    'Total Transactions', 
    'Avg BTC Value', 
    'Total BTC Moved',
    'Avg Fee %',
    'Avg Complexity',
    'Max Transaction',
    'Min Transaction'
]

# Set Year as index
gold_pandas = gold_pandas.set_index('Year')

print("🥇 GOLD LAYER - Bitcoin Analytics Summary")
print("=" * 80)
print(gold_pandas.to_string())

StatementMeta(etlsparkpool, 9, 13, Finished, Available, Finished, False)

🥇 GOLD LAYER - Bitcoin Analytics Summary
      Total Transactions  Avg BTC Value  Total BTC Moved  Avg Fee %  Avg Complexity  Max Transaction  Min Transaction
Year                                                                                                                 
2013              568132          21.88      12433301.21       1.02            1.98         49042.47              0.0
2014              792130          10.63       8419020.44       0.99            2.04         39016.97              0.0
2015             1459810          10.71      15629873.97       1.26            2.78         92114.14              0.0
2016             2628115          12.98      34109669.40       1.40            2.01         61228.00              0.0
2017             3320902           8.15      27058792.64       3.22            2.08         23706.99              0.0
2026                2947           0.63          1853.14       0.47            2.44           152.28              0.0


In [37]:
# # Save cleaned data back to Azure Data Lake
# df_clean.write.mode("overwrite") \
#     .parquet("abfss://btc-transactions@btcetlstorage.dfs.core.windows.net/cleaned/")

# print("✅ Cleaned data saved to Azure Data Lake!")

StatementMeta(etlsparkpool, 9, 14, Finished, Available, Finished, False)

Cleaned Data (Azure)
        ↓
Kafka Producer → reads cleaned parquet
              → sends each transaction as a message
        ↓
Kafka Topic (like a queue) → "btc-transactions"
        ↓
Kafka Consumer → receives messages
              → simulates real-time processing

"📊 STATISTICAL ANALYSIS"

In [38]:
# descriptive statistics
from pyspark.sql.functions import year, count, avg, sum, max, min, stddev

print("📊 STATISTICAL ANALYSIS")
print("=" * 60)
print("\n1️⃣ Descriptive Statistics:")

df_clean.select(
    "output_value", 
    "input_value", 
    "fee", 
    "fee_percentage",
    "complexity_ratio"
).describe().show()

StatementMeta(etlsparkpool, 9, 15, Finished, Available, Finished, False)

📊 STATISTICAL ANALYSIS

1️⃣ Descriptive Statistics:
+-------+------------------+------------------+--------------------+------------------+------------------+
|summary|      output_value|       input_value|                 fee|    fee_percentage|  complexity_ratio|
+-------+------------------+------------------+--------------------+------------------+------------------+
|  count|           8772036|           8772036|             8772036|           8746032|           8772036|
|   mean|11.132251486511576| 11.13274410434898|4.926178373675031E-4|2.0047000929221235|2.1662546336449156|
| stddev|125.24682532854608|125.24695146267803|0.008520744290489865| 7.740812079596978|18.803796943276456|
|    min|               0.0|               0.0|                 0.0|               0.0|            1.0E-4|
|    max|    92114.13874635| 92114.14074634999|                12.2|             100.0|           13107.0|
+-------+------------------+------------------+--------------------+------------------+-----

Interesting Findings:

someone moved 92,114 BTC in one transaction (worth millions!)
someone paid 100% as fee (accidentally sent entire amount as fee )
one transaction had 13,107 receivers (massive distribution!)

 Correlation Analysis!

In [39]:
print("\n2️⃣ Correlation Analysis:")
print("Does higher input_value mean higher fee?")
corr_value_fee = df_clean.stat.corr("input_value", "fee")
print(f"   Correlation (input_value vs fee): {corr_value_fee:.4f}")

print("\nDoes higher output_value mean higher complexity?")
corr_value_complexity = df_clean.stat.corr("output_value", "complexity_ratio")
print(f"   Correlation (output_value vs complexity_ratio): {corr_value_complexity:.4f}")

print("\nDoes higher fee mean higher fee_percentage?")
corr_fee_pct = df_clean.stat.corr("fee", "fee_percentage")
print(f"   Correlation (fee vs fee_percentage): {corr_fee_pct:.4f}")

StatementMeta(etlsparkpool, 9, 16, Finished, Available, Finished, False)


2️⃣ Correlation Analysis:
Does higher input_value mean higher fee?
   Correlation (input_value vs fee): 0.0148

Does higher output_value mean higher complexity?
   Correlation (output_value vs complexity_ratio): 0.0171

Does higher fee mean higher fee_percentage?
   Correlation (fee vs fee_percentage): 0.0117


Key Insight:

Bitcoin fees are NOT correlated with transaction value!

this is actually a famous property of Bitcoin — you can send $1 million for the same fee as sending $1, as long as the transaction data size is the same 

Year over Year Growth!

In [40]:
print("\n3️⃣ Year over Year Growth:")
from pyspark.sql.functions import year, count, lag
from pyspark.sql.window import Window

# Transaction count per year
yearly = df_clean.groupBy(year("block_timestamp").alias("year")) \
    .agg(count("txid").alias("total_transactions")) \
    .orderBy("year")

yearly.show()

StatementMeta(etlsparkpool, 9, 17, Finished, Available, Finished, False)


3️⃣ Year over Year Growth:
+----+------------------+
|year|total_transactions|
+----+------------------+
|2013|            568132|
|2014|            792130|
|2015|           1459810|
|2016|           2628115|
|2017|           3320902|
|2026|              2947|
+----+------------------+



 Year over Year Growth %

In [41]:
import pandas as pd

yearly_pandas = yearly.toPandas()

# Calculate growth percentage
yearly_pandas['growth_%'] = yearly_pandas['total_transactions'].pct_change() * 100
yearly_pandas['growth_%'] = yearly_pandas['growth_%'].round(2)

print("\n📈 Year over Year Transaction Growth:")
print("=" * 50)
print(yearly_pandas.to_string(index=False))

StatementMeta(etlsparkpool, 9, 18, Finished, Available, Finished, False)


📈 Year over Year Transaction Growth:
 year  total_transactions  growth_%
 2013              568132       NaN
 2014              792130     39.43
 2015             1459810     84.29
 2016             2628115     80.03
 2017             3320902     26.36
 2026                2947    -99.91


Key Insights:
2015 had the highest growth at 84.29%!

surprising right? you'd expect 2017
but 2015 was recovery year after Mt.Gox hack in 2014
people regained confidence → massive adoption surge

2017 only grew 26% despite being the bull run year

because 2016 already had 2.6M transactions
growing from a big base is harder than growing from small base

**Fee Trend Analysis! **

In [42]:
print("\n4️ Fee Trend Analysis:")
from pyspark.sql.functions import year, avg

fee_trends = df_clean.groupBy(year("block_timestamp").alias("year")) \
    .agg(
        avg("fee_percentage").alias("avg_fee_pct"),
        avg("fee").alias("avg_fee_btc")
    ).orderBy("year")

fee_trends_pandas = fee_trends.toPandas().round(4)
print(fee_trends_pandas.to_string(index=False))

StatementMeta(etlsparkpool, 9, 19, Finished, Available, Finished, False)


4️ Fee Trend Analysis:
 year  avg_fee_pct  avg_fee_btc
 2013       1.0160       0.0007
 2014       0.9896       0.0002
 2015       1.2607       0.0002
 2016       1.3956       0.0003
 2017       3.2206       0.0008
 2026       0.4699       0.0000


2016 → 1.4% average fee
2017 → 3.2% average fee
→ 130% increase in just one year!

** Whale Activity Analysis! 🐳**

In [43]:
print("\n5️⃣ Whale Activity Analysis:")
from pyspark.sql.functions import year, count, when

whale_analysis = df_clean.groupBy(year("block_timestamp").alias("year")) \
    .agg(
        count("txid").alias("total_transactions"),
        count(when(df_clean.btc_value_category == "whale", 1)).alias("whale_transactions"),
        count(when(df_clean.btc_value_category == "small", 1)).alias("small_transactions")
    ).orderBy("year")

whale_pandas = whale_analysis.toPandas()
whale_pandas["whale_%"] = ((whale_pandas["whale_transactions"] / whale_pandas["total_transactions"]) * 100).round(2)
whale_pandas["small_%"] = ((whale_pandas["small_transactions"] / whale_pandas["total_transactions"]) * 100).round(2)

print(whale_pandas.to_string(index=False))

StatementMeta(etlsparkpool, 9, 20, Finished, Available, Finished, False)


5️⃣ Whale Activity Analysis:
 year  total_transactions  whale_transactions  small_transactions  whale_%  small_%
 2013              568132               31453              320335     5.54    56.38
 2014              792130               25245              550589     3.19    69.51
 2015             1459810               53499              982557     3.66    67.31
 2016             2628115              107131             1788072     4.08    68.04
 2017             3320902               90144             2456489     2.71    73.97
 2026                2947                  12                2833     0.41    96.13


2013 had the HIGHEST whale activity at 5.54%!
this makes total sense:

in 2013 Bitcoin was new
only tech people and big investors knew about it
regular people hadn't adopted it yet
so most transactions were from wealthy early adopters

2017 had the LOWEST whale % at 2.71% despite being the bull run!

because millions of regular retail investors joined
small transactions exploded to 73.97%
whales got "diluted" by massive retail adoption

In [44]:
# Take a sample of 500k rows for Power BI (manageable size)
df_sample = df_clean.sample(fraction=0.06, seed=42)
print(f"Sample size: {df_sample.count()}")

StatementMeta(etlsparkpool, 9, 21, Finished, Available, Finished, False)

Sample size: 526439


In [ ]:
import pandas as pd
from azure.storage.blob import BlobServiceClient

# Convert to pandas and save as CSV
print("Converting to pandas...")
df_pandas = df_sample.toPandas()

print("Saving CSV...")
df_pandas.to_csv("/tmp/btc_clean.csv", index=False)

# Upload to Azure
print("Uploading to Azure...")
conn_str = "DefaultEndpointsProtocol=https;AccountName=btcetlstorage;AccountKey=acc key ;EndpointSuffix=core.windows.net"

blob_service = BlobServiceClient.from_connection_string(conn_str)
blob_client = blob_service.get_blob_client(
    container="btc-transactions", 
    blob="btc_clean.csv"
)

with open("/tmp/btc_clean.csv", "rb") as f:
    blob_client.upload_blob(f, overwrite=True)

print("✅ CSV uploaded to Azure! Ready for Power BI!")

StatementMeta(etlsparkpool, 9, 22, Finished, Available, Finished, False)

Converting to pandas...
Saving CSV...
Uploading to Azure...
✅ CSV uploaded to Azure! Ready for Power BI!


exporting gold as csv to BI

In [1]:
# Read Gold layer (already saved in Azure)
df_gold = spark.read.parquet("abfss://btc-transactions@btcetlstorage.dfs.core.windows.net/gold/")
df_gold.show()

StatementMeta(etlsparkpool, 10, 2, Finished, Available, Finished, False)

+----+------------------+------------------+--------------------+-------------------+--------------------+---------------------+---------------------+
|year|total_transactions|  avg_output_value|  total_output_value| avg_fee_percentage|avg_complexity_ratio|max_transaction_value|min_transaction_value|
+----+------------------+------------------+--------------------+-------------------+--------------------+---------------------+---------------------+
|2013|            568132|21.884528969221744|1.2433301212341888E7|  1.015960537515905|   1.980912580351061|   49042.469999999994|               1.0E-8|
|2014|            792130|10.628331760592749|   8419020.437518334| 0.9895552408064052|    2.03708195308852|             39016.97|                  0.0|
|2015|           1459810|10.706786481291438|1.5629873973254053E7| 1.2606943901057392|   2.784953698358021|       92114.13874635|                  0.0|
|2016|           2628115|12.978758310803123| 3.410966939799635E7| 1.3955737272912045|  2.00806

In [ ]:
import pandas as pd
from azure.storage.blob import BlobServiceClient

gold_pandas = df_gold.toPandas().round(2)

gold_pandas.columns = [
    'Year', 'Total Transactions', 'Avg BTC Value',
    'Total BTC Moved', 'Avg Fee %', 'Avg Complexity',
    'Max Transaction', 'Min Transaction'
]

gold_pandas.to_csv("/tmp/btc_gold.csv", index=False)

conn_str = "DefaultEndpointsProtocol=https;AccountName=btcetlstorage;AccountKey=#acc key;EndpointSuffix=core.windows.net"

blob_service = BlobServiceClient.from_connection_string(conn_str)
blob_client = blob_service.get_blob_client(
    container="btc-transactions",
    blob="btc_gold.csv"
)

with open("/tmp/btc_gold.csv", "rb") as f:
    blob_client.upload_blob(f, overwrite=True)

print("✅ Gold CSV uploaded!")

StatementMeta(etlsparkpool, 10, 3, Finished, Available, Finished, False)

✅ Gold CSV uploaded!
